In [ ]:
import os, time, requests
from typing import List
import jieba

# 引入 LlamaIndex 核心元件
from llama_index.core import StorageContext, Settings, load_index_from_storage, VectorStoreIndex
from llama_index.core.prompts.prompts import SimpleInputPrompt
from llama_index.core.schema import Document
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.node_parser import SentenceSplitter

# 引入 LlamaIndex 的 Ollama 整合
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="jieba")

class LlamaIndexRAG:
    """
    使用 LlamaIndex 的 VectorStoreIndex 實現 RAG 系統
    """
    def __init__(self, model_name="llama3:8b", embed_model_name="mxbai-embed-large", base_url="http://localhost:11434"):
        self.model_name = model_name
        self.embed_model_name = embed_model_name
        self.base_url = base_url
        self.index = None

        # 檢查 Ollama 服務是否運行
        self._check_ollama_status()

        # 配置 LlamaIndex 的 LLM 和嵌入模型
        self.llm = Ollama(model=self.model_name, base_url=self.base_url, request_timeout=30000,
                          additional_kwargs={
                            "num_ctx": 8192,       # prompt總長度限制，避免一次塞太多 token
                            "num_predict": 512     # 限制輸出長度，避免回應太大
                        })
        self.embed_model = OllamaEmbedding(model_name=self.embed_model_name, base_url=self.base_url,
                        request_timeout=600, # 增加超時等待
                        additional_kwargs={
                            "num_ctx": 2048,  # 這裡加到 2048
                            "truncate": True  # 萬一真的太長，叫它自動切掉，不要報錯
                        })

        Settings.llm = self.llm
        Settings.embed_model = self.embed_model

        # 查詢模板
        self.qa_template = SimpleInputPrompt("""
        請參考以下資訊回答問題：
        ---------------------
        {context_str}
        ---------------------
        問題：{query_str}
        請以繁體中文回答，並根據所提供的資訊進行回答。如果資訊不涵蓋問題，請回答：我不知道。
        """)   

    def _check_ollama_status(self):
        """檢查 Ollama 服務器是否可達"""
        try:
            response = requests.get(f"{self.base_url}/api/tags")
            if response.status_code == 200:
                print(f"✅ Ollama 服務器 {self.base_url} 可達。")
            else:
                raise ConnectionError(f"無法連接到 Ollama 服務器。狀態碼: {response.status_code}")
        except requests.exceptions.ConnectionError as e:
            raise ConnectionError(f"無法連接到 Ollama 服務器，請檢查服務是否已啟動: {e}")

    def create_index_from_documents(self, documents: List[str], index_dir: str = "storage"):
            """
            從文檔列表建立 LlamaIndex VectorStoreIndex，並可選擇儲存或從現有索引載入。
            """
            try:
                storage_context = StorageContext.from_defaults(persist_dir=index_dir)
                self.index = load_index_from_storage(storage_context)
                print(f"✅ 成功載入現有索引於 '{index_dir}'。")
            except Exception as e:
                print("📝 找不到現有索引或載入失敗，正在重新建立 VectorStoreIndex...")
                start_time = time.time()
                
                # 將原始字串包裝成 LlamaIndex 的 Document 物件
                docs = [Document(text="".join(documents))] 
                
                # 💡 確保你的大文字切分器邏輯被套用到 Settings 中
                Settings.chunk_size = 512
                Settings.chunk_overlap = 20
                
                # 核心修正：必須傳入 docs 來真正建立 VectorStoreIndex 物件！
                print("⏳ 正在進行 Embedding 並建置 Vector Index (處理長文本小說需要一些時間)...")
                self.index = VectorStoreIndex.from_documents(docs)
                
                # 儲存索引（此時 self.index 已經不是 None，不會再報錯了）
                self.index.storage_context.persist(persist_dir=index_dir)
                print(f"✅ 已成功建立並儲存 VectorStoreIndex 於 '{index_dir}'。")
                print(f"⏱️ 總共花費時間：{time.time() - start_time:.2f} 秒。")
            
    def print_vector_index_structure(self, max_nodes: int = 20):
        """
        列印 VectorStoreIndex 的節點摘要資訊。
        Args: max_nodes: 最多列印的節點數，避免輸出過長
        """
        if self.index is None:
            print("❌ 錯誤：索引尚未建立。")
            return
        print("\n📦 VectorStoreIndex 節點資訊:\n","=" * 50)

        # 從 docstore 取出所有節點
        all_nodes = list(self.index.docstore.docs.values())
        if not all_nodes:
            print("❌ docstore 中沒有任何節點。")
            return
        print(f"共有 {len(all_nodes)} 個節點。")

        for i, node in enumerate(all_nodes[:max_nodes]):
            summary = node.get_content()[:80].replace("\n", " ") + "..."
            file_name = node.metadata.get('file_name', 'N/A')
            page_label = node.metadata.get('page_label', 'N/A')
            print("-" * 50, f"\n[{i+1}] Node ID: {node.node_id}")
            print(f"來源檔案: {file_name}, 頁碼: {page_label}\n",f"摘要: {summary}")

        if len(all_nodes) > max_nodes:
            print(f"... (僅顯示前 {max_nodes} 個節點)")
        print("=" * 50)
    
    def print_vector_index_tokens(self, max_nodes: int = 20, use_jieba: bool = False):
        """
        列印 VectorStoreIndex 節點摘要資訊，並估算每個節點佔用的 token 數。
        
        Args:
            max_nodes: 最多列印的節點數，避免輸出過長
            use_jieba: 若為 True，則以 jieba 分詞計算 token，適合中文
        """
        if self.index is None:
            print("❌ 錯誤：索引尚未建立。")
            return    
        print("\n📦 VectorStoreIndex 節點 token 數資訊:\n", "="*50)    
        all_nodes = list(self.index.docstore.docs.values())
        if not all_nodes:
            print("❌ docstore 中沒有任何節點。")
            return    
        print(f"共有 {len(all_nodes)} 個節點。")    
        for i, node in enumerate(all_nodes[:max_nodes]):
            content = node.get_content().strip()
            # 簡單 token 估算
            if use_jieba:
                import jieba
                tokens = list(jieba.cut(content))
            else:
                # 英文或混合文本，以空格拆分
                tokens = content.split()
            token_count = len(tokens)
    
            summary = content[:80].replace("\n", " ") + "..."
            file_name = node.metadata.get('file_name', 'N/A')
            page_label = node.metadata.get('page_label', 'N/A')
    
            print("-"*50, f"\n[{i+1}] Node ID: {node.node_id}")
            print(f"來源檔案: {file_name}, 頁碼: {page_label}")
            print(f"摘要: {summary}")
            print(f"Token 數 (估算): {token_count}\t Length: {len(content)}")    
        if len(all_nodes) > max_nodes:
            print(f"... (僅顯示前 {max_nodes} 個節點)")
        print("="*50)


    def rag_query(self, query: str, topK=5) -> str:
        """
        對索引進行查詢並獲取 RAG 回答
        """
        if self.index is None:
            return "錯誤：索引尚未建立，無法進行查詢。"
        print("🔍 正在查詢...")
        query_engine = self.index.as_query_engine(
            text_qa_template=self.qa_template,
            similarity_top_k=topK
        )
        try:
            response = query_engine.query(query)
            print(f"🤖 回答: {response.response}\n")
            print("---\n","📄 來源文件 (Source Nodes):")
            if response.source_nodes:
                for i, node in enumerate(response.source_nodes):
                    node_content = node.get_content().strip()
                    print(f"[{i+1}] Node ID: {node.node_id}")
                    print(f"內容摘要: {node_content[:200]}...\n","-" * 20)
            else:
                print("❌ 找不到相關的來源節點。")
            return response.response
        except Exception as e:
            return f"❌ 查詢時發生錯誤：{e}"

# 使用範例
def main(datafile, datafolder,top_k=5, jb: bool = False, 
         model_name="llama3:8b", emodel_name="mxbai-embed-large"        ):
    print("🤖 使用 LlamaIndex VectorStoreIndex 的 RAG 系統測試\n", "=" * 50)

    try:
        rag = LlamaIndexRAG(model_name=model_name, embed_model_name=emodel_name)
    except ConnectionError as e:
        print(f"❌ 錯誤：{e}")
        return

    if not os.path.exists(datafile):
        print(f"❌ 錯誤：找不到文件 '{file_path}'。請確認文件路徑是否正確。")
        return
        
    with open(datafile, "r", encoding="utf-8") as f:
        content = f.read()

    if jb:
        # 中文先分詞
        words = jieba.cut(content)
        content = " ".join(words)
        
    # 將整篇小說直接傳給 create_index_from_documents
    documents = [content]  # LlamaIndex 會自動拆分成多個節點
    
    rag.create_index_from_documents(documents, index_dir=datafolder)

    #rag.print_vector_index_tokens()
    rag.print_vector_index_structure(max_nodes=100)

    print("\n🔍 請輸入查詢問題（輸入 quit 結束）\n", "=" * 60)
    while True:
        query = input("❓ 問題: ").strip()
        if query.lower() == "quit":
            print("\n✅ 結束測試。")
            break
        # 在查詢前記錄開始時間
        start_time = time.time()
        segmented_query = " ".join(jieba.cut(query)) if jb else query
        answer = rag.rag_query(segmented_query, top_k)
        end_time = time.time()
        elapsed_seconds, elapsed_minutes = end_time - start_time, (end_time - start_time)/60
        print(f"🤖 回答: {answer}\n", "-" * 50)
        print(f"⏱️ 查詢花費時間: {elapsed_minutes:.4f} 分鐘 ({elapsed_seconds:.4f} 秒)\n", "-" * 50)
    
if __name__ == "__main__":
    main("pg74.txt", "app5_vector_storage", top_k=10,
        model_name="gpt-oss:20b", emodel_name="mxbai-embed-large" )

🤖 使用 LlamaIndex VectorStoreIndex 的 RAG 系統測試
✅ Ollama 服務器 http://localhost:11434 可達。
📝 找不到現有索引或載入失敗，正在重新建立 VectorStoreIndex...
⏳ 正在進行 Embedding 並建置 Vector Index (處理長文本小說需要一些時間)...
